# Image Fundamentals — Pixels, Channels, Color Spaces Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Load an image and inspect its shape

Use Pillow to load any JPEG or PNG, convert to NumPy, and print what you got. For a deterministic example that runs offline, synthesize one.

In [ ]:
```python

import numpy as np

from PIL import Image

def synthetic_rgb(h=128, w=192, seed=0):

    rng = np.random.default_rng(seed)

    yy, xx = np.meshgrid(np.linspace(0, 1, h), np.linspace(0, 1, w), indexing="ij")

    r = (np.sin(xx * 6) * 0.5 + 0.5) * 255

    g = yy * 255

    b = (1 - yy) * xx * 255

    rgb = np.stack([r, g, b], axis=-1) + rng.normal(0, 6, (h, w, 3))

    return np.clip(rgb, 0, 255).astype(np.uint8)

arr = synthetic_rgb()

# Or load from disk:

# arr = np.asarray(Image.open("your_image.jpg").convert("RGB"))

print(f"type:   {type(arr).__name__}")

print(f"dtype:  {arr.dtype}")

print(f"shape:  {arr.shape}     # (H, W, C)")

print(f"min:    {arr.min()}")

print(f"max:    {arr.max()}")

print(f"pixel at (0, 0): {arr[0, 0]}")

In [ ]:
```

Expected output: `shape: (H, W, 3)`, `dtype: uint8`, range `[0, 255]`. That is the canonical on-disk representation whether the bytes came from a camera, a JPEG decoder, or a synthetic generator.

### Step 2: Split channels and re-order layout

Pull out R, G, B separately, then convert from HWC to CHW for PyTorch.

In [ ]:
```python

R = arr[:, :, 0]

G = arr[:, :, 1]

B = arr[:, :, 2]

print(f"R shape: {R.shape}, mean: {R.mean():.1f}")

print(f"G shape: {G.shape}, mean: {G.mean():.1f}")

print(f"B shape: {B.shape}, mean: {B.mean():.1f}")

arr_chw = arr.transpose(2, 0, 1)

print(f"\nHWC shape: {arr.shape}")

print(f"CHW shape: {arr_chw.shape}")

In [ ]:
```

Three grayscale planes, one per channel. CHW just reorders the axes; no data copy is strictly required when the memory layout allows it.

### Step 3: Grayscale and HSV conversions

Weighted-sum grayscale, then a manual RGB-to-HSV.

In [ ]:
```python

def rgb_to_grayscale(rgb):

    weights = np.array([0.299, 0.587, 0.114], dtype=np.float32)

    return (rgb.astype(np.float32) @ weights).astype(np.uint8)

def rgb_to_hsv(rgb):

    rgb_f = rgb.astype(np.float32) / 255.0

    r, g, b = rgb_f[..., 0], rgb_f[..., 1], rgb_f[..., 2]

    cmax = np.max(rgb_f, axis=-1)

    cmin = np.min(rgb_f, axis=-1)

    delta = cmax - cmin

    h = np.zeros_like(cmax)

    mask = delta > 0

    rmax = mask & (cmax == r)

    gmax = mask & (cmax == g)

    bmax = mask & (cmax == b)

    h[rmax] = ((g[rmax] - b[rmax]) / delta[rmax]) % 6

    h[gmax] = ((b[gmax] - r[gmax]) / delta[gmax]) + 2

    h[bmax] = ((r[bmax] - g[bmax]) / delta[bmax]) + 4

    h = h * 60.0

    s = np.where(cmax > 0, delta / cmax, 0)

    v = cmax

    return np.stack([h, s, v], axis=-1)

gray = rgb_to_grayscale(arr)

hsv = rgb_to_hsv(arr)

print(f"gray shape: {gray.shape}, range: [{gray.min()}, {gray.max()}]")

print(f"hsv   shape: {hsv.shape}")

print(f"hue range: [{hsv[..., 0].min():.1f}, {hsv[..., 0].max():.1f}] degrees")

print(f"sat range: [{hsv[..., 1].min():.2f}, {hsv[..., 1].max():.2f}]")

print(f"val range: [{hsv[..., 2].min():.2f}, {hsv[..., 2].max():.2f}]")

In [ ]:
```

Hue comes out in degrees, saturation and value in [0, 1]. That matches the OpenCV `hsv_full` convention.

### Step 4: Normalize, standardize, and reverse it

Go from raw bytes to the exact tensor a pretrained ImageNet model expects, then back.

In [ ]:
```python

mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)

std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess_imagenet(rgb_uint8):

    x = rgb_uint8.astype(np.float32) / 255.0

    x = (x - mean) / std

    x = x.transpose(2, 0, 1)

    return x

def deprocess_imagenet(chw_float32):

    x = chw_float32.transpose(1, 2, 0)

    x = x * std + mean

    x = np.clip(x * 255.0, 0, 255).astype(np.uint8)

    return x

x = preprocess_imagenet(arr)

print(f"preprocessed shape: {x.shape}     # (C, H, W)")

print(f"preprocessed dtype: {x.dtype}")

print(f"preprocessed mean per channel:  {x.mean(axis=(1, 2)).round(3)}")

print(f"preprocessed std  per channel:  {x.std(axis=(1, 2)).round(3)}")

roundtrip = deprocess_imagenet(x)

max_diff = np.abs(roundtrip.astype(int) - arr.astype(int)).max()

print(f"roundtrip max pixel diff: {max_diff}    # should be 0 or 1")

In [ ]:
```

Per-channel mean should be close to zero, std close to one. The preprocess/deprocess pair is exactly what every torchvision `transforms.Normalize` call is doing under the hood.

### Step 5: Resize with three interpolation methods

Compare nearest, bilinear, and bicubic on an upscale so the difference is visible.

In [ ]:
```python

target = (arr.shape[0] * 3, arr.shape[1] * 3)

nearest = np.asarray(Image.fromarray(arr).resize(target[::-1], Image.NEAREST))

bilinear = np.asarray(Image.fromarray(arr).resize(target[::-1], Image.BILINEAR))

bicubic = np.asarray(Image.fromarray(arr).resize(target[::-1], Image.BICUBIC))

def local_roughness(x):

    gy = np.diff(x.astype(float), axis=0)

    gx = np.diff(x.astype(float), axis=1)

    return float(np.abs(gy).mean() + np.abs(gx).mean())

for name, out in [("nearest", nearest), ("bilinear", bilinear), ("bicubic", bicubic)]:

    print(f"{name:>8}  shape={out.shape}  roughness={local_roughness(out):6.2f}")

In [ ]:
```

Nearest scores highest on roughness because it keeps hard edges. Bilinear is the smoothest. Bicubic sits in between, preserving perceived sharpness without the stair-step artifacts.

## Exercises

In [ ]:
1. **(Easy)** Load a JPEG with OpenCV (`cv2.imread`) and with Pillow. Print both shapes and the pixel at `(0, 0)`. Explain the channel-order difference, then write a one-line conversion that makes the OpenCV array identical to the Pillow one.
2. **(Medium)** Write `standardize(img, mean, std)` and its inverse that together pass a `roundtrip_max_diff <= 1` test on any uint8 image. Your functions must work on a single image in HWC and on a batch in NCHW with the same call.
3. **(Hard)** Take a 3-channel ImageNet-standardized tensor and run it through a 1x1 conv that learns a weighted mixture of RGB into a single grayscale channel. Initialize the weights to `[0.299, 0.587, 0.114]`, freeze them, and verify the output matches your manual `rgb_to_grayscale` to within floating-point error. What other classical color-space transforms can be written as 1x1 convolutions?